# LÉLU reconstruction on a free Kaggle GPU

Zero cost, no payment method, no Hugging Face account required. Kaggle's free
P100 (16 GB) is enough for **TripoSR** (~6 GB VRAM). PSHuman needs >40 GB, so it
cannot run on a P100 — use it only on a 40 GB+ device.

**Before running:** in the right-hand panel choose
`Settings → Accelerator → GPU P100` and `Internet → On`, then `Run All`.

This notebook reconstructs from the canonical A-pose photograph, writes the
mesh plus a preview render, and packages everything into
`/kaggle/working/lelu_recon.zip` for download. Drop that zip's contents into
`leluverse/assets/characters/lelu_character/foundry/cloud/kaggle_<stamp>/` and
run `scripts/foundry/import_cloud_mesh.py` on the GLB to validate it against
the calibrated landmarks.

The reference set is fetched from the project's own GitHub repository, so the
notebook always reconstructs from the same photographs with the same hashes.

In [ ]:
import os, subprocess, sys, json, hashlib, time, shutil
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'],
                     capture_output=True, text=True).stdout or 'NO GPU VISIBLE')

In [ ]:
# 1. Locate the four primary photographs and verify their hashes.
#    Preferred: upload them as a private Kaggle dataset (Add Data -> Upload) named
#    lelu-primary-source. Nothing is pushed to GitHub yet, so the repo URL is only
#    a best-effort fallback.
import glob
WORK = '/kaggle/working/lelu'
os.makedirs(WORK, exist_ok=True)
FILES = ['01_front_full_body.png', '02_face_closeup.png', '03_back_upper_body.png', '04_front_apose.png']
EXPECTED = {'01_front_full_body.png': '1da6e4599cc0', '02_face_closeup.png': 'ca6d058f12b0',
            '03_back_upper_body.png': 'bec0f68f49ed', '04_front_apose.png': '1e01242a0a27'}
REPO = ('https://raw.githubusercontent.com/lapheezy/LeluV1.1/'
        'integration/m16-3-wp05-reconciliation/leluverse/assets/characters/lelu_character/'
        'references/PRIMARY_SOURCE')
missing = []
for f in FILES:
    found = glob.glob(f'/kaggle/input/**/{f}', recursive=True)
    dst = f'{WORK}/{f}'
    if found:
        shutil.copy(found[0], dst)
    else:
        r = subprocess.run(['wget', '-q', f'{REPO}/{f}', '-O', dst])
        if r.returncode != 0 or os.path.getsize(dst) < 1000:
            missing.append(f); continue
    h = hashlib.sha256(open(dst, 'rb').read()).hexdigest()
    flag = 'OK' if h.startswith(EXPECTED[f]) else 'HASH MISMATCH'
    print(f'{f:28s} {os.path.getsize(dst)/1e6:5.2f} MB  {h[:16]}  {flag}')
if missing:
    raise SystemExit('Missing references: ' + ', '.join(missing) +
                     '\nUpload them as a Kaggle dataset (Add Data -> Upload -> lelu-primary-source) '
                     'or place them in /kaggle/working/lelu before running.')
print('all four primary references present')

In [ ]:
# 2. TripoSR: open source (MIT), weights from a public repo, no token.
%cd /kaggle/working
!git clone -q https://github.com/VAST-AI-Research/TripoSR.git
%cd TripoSR
!pip install -q -r requirements.txt
!pip install -q trimesh rembg onnxruntime
print('installed')

In [ ]:
# 3. Reconstruct the full body from the A-pose photograph.
%cd /kaggle/working/TripoSR
OUT = '/kaggle/working/lelu/triposr_apose'
t0 = time.time()
!python run.py /kaggle/working/lelu/04_front_apose.png --output-dir {OUT} --model-save-format glb --foreground-ratio 0.90 --mc-resolution 256 --bake-texture --texture-resolution 2048 --render
print('wall clock %.1fs' % (time.time() - t0))
print(sorted(os.listdir(OUT)) if os.path.isdir(OUT) else 'NO OUTPUT DIR')

In [ ]:
# 4. Head/face pass at higher mesh resolution, for the identity geometry.
%cd /kaggle/working/TripoSR
OUT_FACE = '/kaggle/working/lelu/triposr_face'
!python run.py /kaggle/working/lelu/02_face_closeup.png --output-dir {OUT_FACE} --model-save-format glb --foreground-ratio 0.85 --mc-resolution 320
print(sorted(os.listdir(OUT_FACE)) if os.path.isdir(OUT_FACE) else 'NO OUTPUT DIR')

In [ ]:
# 5. Report what actually came back: triangles, bounds, preview image.
import trimesh, glob
report = {'created': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'assets': {}}
for name, d in (('body', OUT), ('face', OUT_FACE)):
    files = glob.glob(f'{d}/**/*.glb', recursive=True) + glob.glob(f'{d}/**/*.obj', recursive=True)
    if not files:
        report['assets'][name] = {'note': 'no mesh produced'}
        continue
    p = sorted(files, key=os.path.getsize)[-1]
    m = trimesh.load(p, force='mesh')
    report['assets'][name] = {'file': os.path.basename(p), 'bytes': os.path.getsize(p),
                              'triangles': int(len(m.faces)), 'bounds': m.bounds.round(4).tolist(),
                              'watertight': bool(m.is_watertight)}
print(json.dumps(report, indent=2))
json.dump(report, open('/kaggle/working/lelu/cloud_run_report.json', 'w'), indent=2)

In [ ]:
# 6. Package everything so the only copy is never inside the notebook.
import shutil
shutil.make_archive('/kaggle/working/lelu_recon', 'zip', '/kaggle/working/lelu')
for f in glob.glob('/kaggle/working/lelu/**/*.*', recursive=True):
    print(f'{os.path.getsize(f)/1e6:7.2f} MB  {f}')
print('\nDownload lelu_recon.zip from the notebook Output panel, then import it:')
print("  python scripts/foundry/import_cloud_mesh.py '{\"mesh\": \".../body.glb\", "
      "\"outDir\": \".../foundry/cloud/kaggle_import\", \"height\": 1.72}'")
print('The import step normalises height, checks landmarks, renders front/back/profile/three-quarter')
print('and writes compare.png against PRIMARY_SOURCE. A mesh that fails that gate is not accepted.')

## What this notebook proves, and what it does not

- It runs a real image-to-3D model on free hardware and writes files you can
  download, with no purchase and no paid API.
- It does **not** by itself prove fidelity. The acceptance test is the import
  gate in the repository: proportions against the calibrated landmarks
  (hairline 0.0305, eye 0.0662, chin 0.1492, shoulder 0.1754, bust 0.2216,
  waist 0.2805, belt 0.3277, crotch 0.4580, knee 0.7350, ankle 0.9000 of
  figure height) and four-view comparison sheets.
- TripoSR output is a single-shell reconstruction: expect fused fingers, no
  separation between hair and clothing, and no rig. It is a base to refine in
  Blender, not a finished character.
- PSHuman needs >40 GB VRAM and cannot run on this P100. Run that route on a
  free 40 GB-tier ZeroGPU Space (with a free HF token), not here.

## Repeated runs

Kaggle gives limited free GPU hours per week. Each run is independent and
writes `cloud_run_report.json`; append it to
`foundry/cloud/runs.jsonl` when you import the results so the GPU usage,
runtime and output hashes stay with the project.